In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Knowledge Base
documents = [
    {"id": "HR-001", "text": "Employees receive 26 annual leave days."},
    {"id": "HR-002", "text": "Leave requests must be approved by the manager."},
    {"id": "HR-003", "text": "Employees have supplementary health insurance."},
    {"id": "HR-004", "text": "Monthly salaries are paid at the end of each month."},
    {"id": "HR-005", "text": "Employees can work remotely two days per week."},
    {"id": "HR-006", "text": "Employees can request unpaid leave."},
]

# 2. Load Embedding Model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Generate Document Embeddings
document_texts = [doc["text"] for doc in documents]

document_embeddings = model.encode(
    document_texts, convert_to_numpy=True, normalize_embeddings=True
)


# 4. Retrieve Relevant Chunks
def retrieve_chunks(query, top_k=3):

    query_embedding = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append(
            {
                "id": documents[index]["id"],
                "text": documents[index]["text"],
                "score": float(scores[index]),
            }
        )

    return results


# 5. Construct Context
def construct_context(retrieved_chunks):

    context_parts = []

    for chunk in retrieved_chunks:
        context_parts.append(f"[Source: {chunk['id']}]\n{chunk['text']}")

    return "\n\n".join(context_parts)


# 6. Construct RAG Prompt
def construct_prompt(query, context):

    system_instruction = """
You are an HR assistant.

Rules:
- Answer using only the provided context.
- Do not invent or assume information.
- If the context does not contain the answer,
  clearly say that you don't know.
- Keep your answer clear and concise.
"""

    prompt = f"""
{system_instruction}

Context:
{context}

User Question:
{query}

Answer:
"""

    return prompt.strip()


# 7. Execute Pipeline
query = "How many annual leave days do employees receive?"

retrieved_chunks = retrieve_chunks(query, top_k=3)

context = construct_context(retrieved_chunks)

prompt = construct_prompt(query, context)

# 8. Display Prompt
print(prompt)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1012.04it/s]


You are an HR assistant.

Rules:
- Answer using only the provided context.
- Do not invent or assume information.
- If the context does not contain the answer,
  clearly say that you don't know.
- Keep your answer clear and concise.


Context:
[Source: HR-001]
Employees receive 26 annual leave days.

[Source: HR-006]
Employees can request unpaid leave.

[Source: HR-005]
Employees can work remotely two days per week.

User Question:
How many annual leave days do employees receive?

Answer:
